In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 4. Local Resume Rewriter

## What We're Building
A tool that takes rough resume bullet points and rewrites them to be:
- **Impact-focused** (results, not just responsibilities)
- **Quantified** where possible
- **Action-verb led** (strong first words)
- **Tailored** to a target role

**You will learn:**
- Few-shot prompting with real examples
- Iterative prompt refinement
- Output comparison and quality scoring

**Stack:** Ollama, LangChain

In [ ]:
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.3)
print("LLM ready!")

## Step 1 — Define the Rewriting Strategy with Few-Shot Examples

Few-shot prompting gives the LLM concrete examples of the transformation
we want — this dramatically improves consistency.

In [ ]:
# Few-shot examples showing weak → strong bullet transformations
examples = [
    {
        "input": "Responsible for managing a team of developers",
        "output": "Led a team of 8 developers, delivering 3 major product releases on schedule and reducing sprint overruns by 25%"
    },
    {
        "input": "Worked on the company's website",
        "output": "Redesigned company website using React and TypeScript, improving page load time by 40% and increasing user engagement by 15%"
    },
    {
        "input": "Helped with data analysis projects",
        "output": "Built automated data pipelines using Python and SQL, processing 2M+ records daily and reducing manual analysis time from 8 hours to 15 minutes"
    },
    {
        "input": "Did customer support and fixed issues",
        "output": "Resolved 50+ customer escalations monthly with 95% satisfaction rate, identifying recurring issues that led to 3 product improvements"
    },
]

# Create the few-shot prompt template
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Rewrite this resume bullet: {input}"),
    ("ai", "{output}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Full prompt with system message + few-shot examples
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert resume writer. Transform weak resume bullets into
powerful, impact-focused statements. Rules:
1. Start with a strong action verb
2. Include quantifiable results where possible
3. Show impact, not just responsibility
4. Keep each bullet to 1-2 lines
5. If quantification isn't possible, describe scope/scale instead"""),
    few_shot_prompt,
    ("human", "Rewrite this resume bullet: {input}"),
])

rewrite_chain = rewrite_prompt | llm | StrOutputParser()
print("Rewrite chain ready!")

## Step 2 — Rewrite Single Bullets

In [ ]:
# Test with weak bullets
weak_bullets = [
    "Managed social media accounts",
    "Wrote Python scripts for the team",
    "Participated in code reviews",
    "Was responsible for the CI/CD pipeline",
    "Helped onboard new employees",
]

print("=== Resume Bullet Rewriter ===\n")
for bullet in weak_bullets:
    rewritten = rewrite_chain.invoke({"input": bullet})
    print(f"BEFORE: {bullet}")
    print(f"AFTER:  {rewritten}")
    print()

## Step 3 — Tailor Bullets to a Target Role

In [ ]:
# Role-specific rewriting
tailor_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert resume writer. Rewrite the resume bullet to be
maximally relevant for the TARGET ROLE. Emphasize skills and achievements
that align with the job requirements. Keep it to 1-2 lines."""),
    ("human", """TARGET ROLE: {target_role}

Original bullet: {bullet}

Rewrite this bullet to be highly relevant for the target role:"""),
])

tailor_chain = tailor_prompt | llm | StrOutputParser()

# Example: same bullet, different target roles
original = "Built and maintained multiple web applications using various technologies"

roles = [
    "Senior Frontend Engineer at a fintech company",
    "DevOps Engineer at a cloud infrastructure startup",
    "Technical Product Manager at an enterprise SaaS company",
]

print(f"Original: {original}\n")
for role in roles:
    tailored = tailor_chain.invoke({"target_role": role, "bullet": original})
    print(f"For '{role}':")
    print(f"  → {tailored}\n")

## Step 4 — Batch Resume Section Rewriter

In [ ]:
def rewrite_resume_section(bullets: list[str], section_name: str = "Experience",
                          target_role: str = None) -> list[str]:
    """Rewrite an entire resume section."""
    print(f"\n{'='*50}")
    print(f"Section: {section_name}")
    if target_role:
        print(f"Target: {target_role}")
    print(f"{'='*50}\n")

    results = []
    for i, bullet in enumerate(bullets, 1):
        if target_role:
            rewritten = tailor_chain.invoke({"target_role": target_role, "bullet": bullet})
        else:
            rewritten = rewrite_chain.invoke({"input": bullet})
        results.append(rewritten)
        print(f"  {i}. ORIGINAL: {bullet}")
        print(f"     REWRITE: {rewritten}\n")
    return results

# Example resume section
experience_bullets = [
    "Managed the development of new features",
    "Worked with cross-functional teams",
    "Improved system performance",
    "Wrote unit tests for the backend",
    "Presented project updates to stakeholders",
]

improved = rewrite_resume_section(
    experience_bullets,
    section_name="Senior Software Engineer — Acme Corp",
    target_role="Staff Engineer at a growth-stage startup"
)

## Summary & Next Steps

**What you learned:**
- ✅ Few-shot prompting for consistent transformations
- ✅ Role-aware content tailoring
- ✅ Batch processing with chains
- ✅ Prompt engineering for creative writing tasks

**Next steps:**
- Add a quality scoring step  (Project 62 covers LLM-as-judge)
- Generate multiple variants and let the user pick
- Combine with Project 5 (Cover Letter Generator) for full job application flow

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

prompt = PromptTemplate.from_template(
    "You are a helpful local AI assistant. Answer the user's question:\n\nQuestion: {question}\n\nAnswer:"
)

chain = prompt | llm | StrOutputParser()

# response = chain.invoke({"question": "What can you help me with?"})
# print(response)
print("LangChain inference pipeline ready!")
